In [ ]:
import requests
import xarray as xr
import pandas as pd
from pathlib import Path

# --- MidCal bbox ---
lat_min, lat_max = 36.0, 37.5
lon_min, lon_max = -124.0, -122.0

# --- paths (from 2_NOTEBOOKS/*) ---
tmp_nc  = Path("../../1_DATA/raw/oisst_midcal_bbox_monthly_tmp.nc")
out_csv = Path("../../1_DATA/processed/oisst_midcal_bbox_monthly.csv")
tmp_nc.parent.mkdir(parents=True, exist_ok=True)
out_csv.parent.mkdir(parents=True, exist_ok=True)

# --- NOAA PSL monthly OISST (NCSS subset) ---
time_start = "1981-09-01T00:00:00Z"
time_end   = pd.Timestamp.today().strftime("%Y-%m-01T00:00:00Z")

url = (
    "https://psl.noaa.gov/thredds/ncss/grid/"
    "Datasets/noaa.oisst.v2.highres/sst.mon.mean.nc"
    f"?var=sst&north={lat_max}&south={lat_min}&west={lon_min}&east={lon_max}"
    "&disableProjSubset=on&horizStride=1"
    f"&time_start={time_start}&time_end={time_end}&timeStride=1"
    "&accept=netcdf4"
)

print("Requesting:", url)
r = requests.get(url, timeout=180)
r.raise_for_status()
tmp_nc.write_bytes(r.content)
print("Downloaded:", tmp_nc.resolve(), "bytes:", tmp_nc.stat().st_size)

ds = xr.open_dataset(tmp_nc)

# coords are usually lat/lon/time on this dataset
lat_name  = "lat" if "lat" in ds.coords else "latitude"
lon_name  = "lon" if "lon" in ds.coords else "longitude"
time_name = "time"

sst = ds["sst"].mean(dim=[lat_name, lon_name], skipna=True)

sst_m = sst.to_series()
sst_m.index = pd.to_datetime(sst_m.index)
sst_m = sst_m.sort_index()
sst_m.name = "sst"

# anomaly vs fixed baseline (good practice): 1991–2020 if available
baseline = sst_m.loc["1991-01-01":"2020-12-01"] if len(sst_m.loc["1991":"2020"]) else sst_m
clim = baseline.groupby(baseline.index.month).mean()
sst_anom_m = sst_m - sst_m.index.month.map(clim)

df_out = pd.DataFrame({"sst": sst_m, "sst_anom": sst_anom_m})
df_out.to_csv(out_csv, index=True)

print("Saved:", out_csv.resolve())
print("Range:", df_out.index.min(), "to", df_out.index.max(), "rows:", len(df_out))
print(df_out.head())